# QFT and phase estimation

Estimate a phase exactly representable with three counting qubits and compare its full output distribution.

## What you will learn

- How to express this workflow with Qiskit's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    print_scaling_table,
    qiskit_selection,
    total_variation_distance,
)

## 1. Define the quantum problem

Quantum phase estimation stores the eigenphase in a counting register and decodes it with an inverse QFT.

In [2]:
from qiskit.circuit.library import QFT

counting = 3
phase = 3 / 8
circuit = QuantumCircuit(counting + 1)
circuit.x(counting)
circuit.h(range(counting))
for wire in range(counting):
    circuit.cp(2 * np.pi * phase * (2 ** wire), wire, counting)
circuit.append(QFT(counting, inverse=True, do_swaps=True).to_gate(), range(counting))

def get_reference():
    return Statevector.from_instruction(circuit).probabilities(qargs=range(counting))

/var/folders/tm/6bh1bn3x6pgfgp8nvpknylq40000gn/T/ipykernel_6056/1865013224.py:10: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  circuit.append(QFT(counting, inverse=True, do_swaps=True).to_gate(), range(counting))


## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(get_reference)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
backend = MettleQBackend(method="statevector", device="cpu")
compiled = transpile(circuit, backend, optimization_level=1)

def get_mettleq():
    state = backend.run(compiled, shots=1, return_statevector=True).result().data(0)["statevector"]
    return Statevector(state).probabilities(qargs=range(counting))

candidate, mettleq_ms, _ = benchmark(get_mettleq)
error = max_abs_error(reference, candidate)
reference_mode = int(np.argmax(reference))
candidate_mode = int(np.argmax(candidate))
method, device = qiskit_selection(backend)

## 4. Check correctness before discussing speed

We compare the counting-register distribution and require the same modal phase estimate.

In [5]:
tutorial_result = emit_result(
    notebook="qiskit/08_qft_and_phase_estimation.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="phase-register probabilities atol=2e-6 and identical mode",
    passed=error <= 2e-6 and candidate_mode == reference_mode,
    exact_match=candidate_mode == reference_mode,
    selected_method=method,
    selected_device=device,
    metrics={"max_probability_error": error, "expected_phase": phase, "reference_mode": reference_mode, "mettleq_mode": candidate_mode},
)


Comparison summary
------------------
Correctness contract: PASS — phase-register probabilities atol=2e-6 and identical mode
SDK reference median: 1.538 ms
MettleQ median:       0.650 ms
Timing interpretation: MettleQ was 2.367x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: yes

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "phase-register probabilities atol=2e-6 and identical mode", "exact_match": true, "framework": "qiskit", "machine": "arm64", "metrics": {"expected_phase": 0.375, "max_probability_error": 2.3841856400252937e-07, "mettleq_mode": 3, "reference_mode": 3}, "mettleq_median_ms": 0.6497499998658895, "notebook": "qiskit/08_qft_and_phase_estimation.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 1.5377920062746853, "reference_over_mettleq": 2.3667441425041793, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}


## What should you conclude?

Use this structure for local ideal QPE studies; benefit depends on total register width and circuit depth.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.